# Polymarket History — Naive Backtest with pandas

This notebook shows how to load the `candles_1d` Parquet data into pandas and run
a simple momentum strategy on Polymarket binary markets.

**Point-in-time discipline:** all signals and position entry prices are gated on
`released_at <= signal_date` — we never look at data that wouldn't have been
available to a trader at signal time.

**Attribution:** Data sourced from Polymarket via Pancake (usepancake.com)


In [1]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

REPO_ROOT = Path('..').resolve()
DATASETS = REPO_ROOT / 'datasets' / 'v1' / 'polymarket'

con = duckdb.connect(':memory:')

for table in ['markets', 'candles_1d', 'resolutions']:
    pq = str(DATASETS / table / '**' / '*.parquet').replace('\\', '/')
    try:
        con.execute(f"CREATE OR REPLACE VIEW {table} AS SELECT * FROM read_parquet('{pq}', hive_partitioning=false)")
        n = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        print(f'{table}: {n:,} rows')
    except Exception as e:
        print(f'{table}: {e}')

markets: 24,304 rows
candles_1d: 851,131 rows
resolutions: 7,788 rows


## Load candles with point-in-time gating

We load all YES-side daily candles for resolved binary markets.

Key constraint: each bar's `released_at` timestamp is the earliest time a consumer
may use it. A live strategy **must** gate on `released_at <= signal_date`.


In [2]:
# Load resolved binary markets with enough bars for a strategy
candles_df = con.execute("""
    WITH resolved_binary AS (
        SELECT m.market_id, m.title, m.category, r.resolved_at
        FROM markets m
        JOIN resolutions r ON m.market_id = r.market_id
        WHERE m.market_kind = 'binary'
    ),
    market_bar_counts AS (
        SELECT c.market_id, COUNT(*) AS n_bars
        FROM candles_1d c
        WHERE c.outcome_label ILIKE '%yes%'
        GROUP BY c.market_id
        HAVING COUNT(*) >= 30  -- at least 30 days of data for meaningful signals
    )
    SELECT
        c.market_id,
        rb.title,
        rb.category,
        rb.resolved_at,
        c.observed_at,
        c.released_at,
        c.open,
        c.high,
        c.low,
        c.close,
        c.volume
    FROM candles_1d c
    JOIN resolved_binary rb ON c.market_id = rb.market_id
    JOIN market_bar_counts mbc ON c.market_id = mbc.market_id
    WHERE c.outcome_label ILIKE '%yes%'
    ORDER BY c.market_id, c.observed_at
""").df()

print(f'Candles loaded: {len(candles_df):,}')
print(f'Unique markets: {candles_df.market_id.nunique():,}')
print(f'Date range: {candles_df.observed_at.min()} — {candles_df.observed_at.max()}')

Candles loaded: 378,642
Unique markets: 2,668
Date range: 1668729624 — 1779235215


## Naive momentum strategy

**Signal:** if the YES price is above its 14-day simple moving average, go long
(buy YES). Otherwise stay flat.

**Entry/exit:** enter at the next bar's `open` after the signal. Exit when the
market resolves (close price on resolution bar) or when signal flips negative.

**Point-in-time gate:** we only use bars where `released_at <= bar_date` (i.e.,
we do not peek at data that arrived late).


In [3]:
def backtest_market(df: pd.DataFrame, sma_window: int = 14) -> dict:
    """
    Naive SMA momentum backtest for one market's YES-side candles.
    Returns a dict of performance metrics.
    """
    df = df.sort_values('observed_at').copy()
    
    # Point-in-time gate: only use bars where released_at <= observed_at
    # (released_at >= observed_at per contract; if released_at > observed_at
    # that bar arrived late and must not be used on the bar's date)
    df = df[df['released_at'] <= df['observed_at'] + 86400].copy()  # 1-day tolerance
    
    if len(df) < sma_window + 2:
        return None
    
    df['sma'] = df['close'].rolling(sma_window).mean()
    df['signal'] = (df['close'] > df['sma']).astype(int)  # 1=long, 0=flat
    
    # Daily return: next-bar open vs this-bar open
    df['next_open'] = df['open'].shift(-1)
    df['bar_return'] = (df['next_open'] / df['open'] - 1).clip(-0.5, 0.5)
    
    # Strategy return: signal from previous bar applied to this bar's return
    df['strat_return'] = df['signal'].shift(1).fillna(0) * df['bar_return']
    
    n_days = len(df)
    cumulative_return = (1 + df['strat_return'].fillna(0)).prod() - 1
    bnh_return = df['close'].iloc[-1] / df['close'].iloc[0] - 1
    
    sharpe = (
        df['strat_return'].mean() / df['strat_return'].std() * np.sqrt(252)
        if df['strat_return'].std() > 0 else 0
    )
    
    return {
        'n_days': n_days,
        'cumulative_return': round(cumulative_return, 4),
        'bnh_return': round(bnh_return, 4),
        'sharpe': round(sharpe, 2),
        'n_long_days': int(df['signal'].shift(1).fillna(0).sum()),
    }

# Run across all markets
results = []
for market_id, group in candles_df.groupby('market_id'):
    r = backtest_market(group)
    if r:
        r['market_id'] = market_id
        r['title'] = group['title'].iloc[0]
        r['category'] = group['category'].iloc[0]
        results.append(r)

results_df = pd.DataFrame(results).sort_values('cumulative_return', ascending=False)
print(f'Markets backtested: {len(results_df):,}')
print(f'Avg strategy return: {results_df.cumulative_return.mean():.2%}')
print(f'Avg Sharpe ratio:    {results_df.sharpe.mean():.2f}')
print()
print('Top 10 by strategy return:')
print(results_df[['title', 'category', 'n_days', 'cumulative_return', 'bnh_return', 'sharpe']].head(10).to_string(index=False))

Markets backtested: 2,668
Avg strategy return: -27.85%
Avg Sharpe ratio:    -0.56

Top 10 by strategy return:
                                                                                          title category  n_days  cumulative_return  bnh_return  sharpe
                                      Will A Minecraft Movie be the top grossing movie of 2025?    other     370            43.6006     32.8814    2.35
                                           Will Shigeru Ishiba be the first leader out in 2025?    other     224            13.1053     18.5800    2.50
                                           Will JD Vance win the 2024 Republican VP nomination? politics     173            13.0798     11.4938    2.95
                                            Will the Indiana Pacers win the Eastern Conference?   sports     240            11.6965     17.9500    2.28
                                   Will Catherine Connolly win the Irish Presidential Election? politics     219             7.909

In [4]:
# Distribution summary
print('Strategy return distribution:')
print(results_df['cumulative_return'].describe().round(4))
print()
print('% of markets where strategy beat buy-and-hold:')
beat = (results_df['cumulative_return'] > results_df['bnh_return']).mean()
print(f'  {beat:.1%}')
print()
print('Naive SMA momentum has limited edge here — prediction market prices')
print('are generally well-calibrated and mean-revert at extremes.')
print('For more sophisticated strategies with zero setup, see usepancake.com')

Strategy return distribution:
count    2668.0000
mean       -0.2785
std         1.2102
min        -0.9996
25%        -0.7763
50%        -0.4940
75%         0.0000
max        43.6006
Name: cumulative_return, dtype: float64

% of markets where strategy beat buy-and-hold:
  79.3%

Naive SMA momentum has limited edge here — prediction market prices
are generally well-calibrated and mean-revert at extremes.
For more sophisticated strategies with zero setup, see usepancake.com


## Key takeaway

This naive SMA momentum strategy shows limited consistent edge across Polymarket 
binary markets. Prediction market prices are generally efficient and well-calibrated.

More sophisticated strategies might exploit:
- **Information arrival events** (e.g., price jumps around news)
- **Cross-market correlation** (correlated binary markets)
- **Resolution timing patterns** (markets resolving faster than expected)
- **Category-specific dynamics** (politics vs. crypto vs. sports)

---

## Want to do this with zero data wrangling?

Visit **[usepancake.com](https://usepancake.com)** — describe your strategy idea in plain English,
Claude builds it, Pancake runs a rigorous backtest with correct point-in-time semantics,
and you get a verifiable, shareable result in minutes.

---
*Attribution: Data sourced from Polymarket via Pancake (usepancake.com) — CC BY 4.0*
